# Capstone 2 : the harness, by hand

The first capstone used a framework. This one uses nothing : a model, four
tools, and a loop we write ourselves.

That loop, with tools that read and write files and run code, is the whole of
Claude Code, Codex and every other coding agent. What they add on top is
visible by the end of this notebook, and it is about forty lines.

Run the cells in order and watch. There is nothing to write.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## The workspace

A folder with a small program, a file of checks that currently fail, and three
sales exports. All made up, and the agent only ever sees them through tools.

In [ ]:
import shutil
from pathlib import Path

WORKSPACE = Path("workspace").resolve()

PROGRAM = """def line_total(price, quantity):
    return price + quantity


def invoice_total(lines):
    return sum(line_total(p, q) for p, q in lines)
"""

CHECKS = """from invoice import invoice_total

lines = [(10, 2), (5, 4)]      # two of something at 10, four of something at 5
got = invoice_total(lines)
assert got == 40, f"expected 40, got {got}"
print("all checks pass")
"""

SALES = {
    "north": [("2026-03-01", "widget", 120.0), ("2026-03-02", "gadget", 80.5),
              ("2026-03-05", "widget", 99.5)],
    "south": [("2026-03-01", "gadget", 200.0), ("2026-03-03", "widget", 45.0)],
    "east":  [("2026-03-02", "widget", 310.0), ("2026-03-04", "gadget", 15.0),
              ("2026-03-06", "gadget", 75.0)],
}


def make_workspace():
    """Start again from a clean folder."""
    shutil.rmtree(WORKSPACE, ignore_errors=True)
    (WORKSPACE / "sales").mkdir(parents=True)
    (WORKSPACE / "invoice.py").write_text(PROGRAM)
    (WORKSPACE / "check.py").write_text(CHECKS)
    for region, rows in SALES.items():
        (WORKSPACE / "sales" / f"{region}.csv").write_text(
            "date,product,amount\n" + "".join(f"{d},{p},{a}\n" for d, p, a in rows))


make_workspace()
for p in sorted(WORKSPACE.rglob("*")):
    if p.is_file():
        print(p.relative_to(WORKSPACE).as_posix())

## Four tools

Ordinary Python functions. Every path is kept inside the workspace, and the
model never sees these bodies ; it only sees what they return.

In [ ]:
import subprocess
import sys


def _inside(path):
    """Resolve a path and refuse anything outside the workspace."""
    p = (WORKSPACE / path).resolve()
    if WORKSPACE not in p.parents:
        raise ValueError(f"{path} is outside the workspace")
    return p


def list_files():
    files = [p for p in sorted(WORKSPACE.rglob("*"))
             if p.is_file() and "__pycache__" not in p.parts]
    return "\n".join(p.relative_to(WORKSPACE).as_posix() for p in files)


def read_file(path):
    return _inside(path).read_text()


def write_file(path, content):
    p = _inside(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"wrote {len(content)} characters to {path}"


def run_python(path):
    r = subprocess.run([sys.executable, str(_inside(path))], cwd=WORKSPACE,
                       capture_output=True, text=True, timeout=30)
    return f"exit code {r.returncode}\n" + (r.stdout + r.stderr).strip()[-2000:]

Call two of them ourselves first. No model involved yet.

In [ ]:
print(list_files())
print()
print(run_python("check.py"))

## What the model is told

Not the code. A name, one sentence, and the names of the parameters. This is
the entry for `read_file`, and it travels with every request we make.

In [ ]:
READ_FILE = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Return the contents of a file in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
}

The same for the other three, and a table from name to function so we can run whatever comes back.

In [ ]:
SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List every file in the workspace.",
        "parameters": {"type": "object", "properties": {}}}},
    READ_FILE,
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Create or overwrite a file with the given content.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"},
                                      "content": {"type": "string"}},
                       "required": ["path", "content"]}}},
    {"type": "function", "function": {
        "name": "run_python",
        "description": "Run a Python file in the workspace and return its output and exit code.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
]

TOOLS = {"list_files": list_files, "read_file": read_file,
         "write_file": write_file, "run_python": run_python}

## One call, no loop

Send the task and the tool list, and look at what comes back. It is not an
answer. The model asks us to run a tool.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE, api_key=KEY)

SYSTEM = ("You work in a small folder of files. Look before you act, check "
          "your work by running it, and finish with a short summary of what "
          "you did.")

TASK_A = ("The checks in check.py fail. Find out why, fix the program, and "
          "run the checks again to prove it.")

messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": TASK_A}]

reply = client.chat.completions.create(model=MODEL, messages=messages,
                                       tools=SCHEMA, temperature=0,
                                       extra_body=THINKING)
message = reply.choices[0].message

print("content   :", repr(message.content))
for call in message.tool_calls or []:
    print("tool call :", call.function.name, call.function.arguments)

Do what it asked, put the result in the conversation, and call again. Two
rules : the model's own request stays in the history, and the result is a
message with the role `tool`.

In [ ]:
import json

messages.append(message)
for call in message.tool_calls:
    args = json.loads(call.function.arguments)
    result = TOOLS[call.function.name](**args)
    print(f"{call.function.name}({args}) ->\n{result}\n")
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

reply = client.chat.completions.create(model=MODEL, messages=messages,
                                       tools=SCHEMA, temperature=0,
                                       extra_body=THINKING)
message = reply.choices[0].message

print("content   :", repr(message.content))
for call in message.tool_calls or []:
    print("next it wants :", call.function.name, call.function.arguments)

We could keep doing this by hand. A loop does it for us, and that loop is
the agent.

## The loop

Call the model. If it asks for tools, run them, add the results, go round
again. If it does not, that is the answer.

In [ ]:
def agent(task):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    while True:
        reply = client.chat.completions.create(model=MODEL, messages=messages,
                                               tools=SCHEMA, temperature=0,
                                               extra_body=THINKING)
        message = reply.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            result = TOOLS[call.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": result})

Fresh workspace, then let it go. This takes about fifteen seconds.

In [ ]:
make_workspace()
print(agent(TASK_A))
print()
print(run_python("check.py"))

It listed the folder, read both files, ran the checks to see them fail,
rewrote the program, and ran the checks again. We did not give it that
order ; it chose it.

That loop is also missing four things, and module 09 was about
every one of them :

- nothing stops it if the model never stops asking
- a tool that raises an exception takes the whole loop down with it
- it will write files and run code without asking anyone
- we cannot see what it did, or what it cost

## The harness

The same loop, with those four added. Each one is marked.

In [ ]:
def brief(args):
    """Arguments, shortened enough to print on one line."""
    return ", ".join(f"{k}={str(v)[:30]!r}" for k, v in args.items())


def run_agent(task, max_turns=15, approve=None):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    tokens = 0
    for turn in range(1, max_turns + 1):                            # 1. a limit
        reply = client.chat.completions.create(model=MODEL, messages=messages,
                                               tools=SCHEMA, temperature=0,
                                               extra_body=THINKING)
        message = reply.choices[0].message
        tokens += reply.usage.total_tokens
        messages.append(message)
        print(f"turn {turn:2}  request {reply.usage.prompt_tokens:,} tokens")  # 4. a meter
        if not message.tool_calls:
            print(f"\n{turn} turns, {tokens:,} tokens in total\n")
            return message.content
        for call in message.tool_calls:
            name, args = call.function.name, json.loads(call.function.arguments)
            if approve and not approve(name, args):                 # 3. asking first
                result = (f"{name} was refused by the person supervising you. "
                          "Do not try another route ; finish now and say what you could not do.")
            else:
                try:
                    result = TOOLS[name](**args)
                except Exception as e:                              # 2. errors go back
                    result = f"ERROR {type(e).__name__}: {e}"
            first = (result.splitlines() or [""])[0][:60]
            print(f"          {name}({brief(args)})  ->  {first}")  # 4. a transcript
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": result})
    print(f"\nstopped after {max_turns} turns, {tokens:,} tokens in total\n")
    return None

The same task again, and this time we can watch.

In [ ]:
make_workspace()
print(run_agent(TASK_A))

Look at the `request` column. Every turn sends the whole conversation
again, so the request grows with every tool result it carries. That is the
context budget from module 04, being spent.

## Same loop, different task

Nothing in the harness changes. Only the task does.

In [ ]:
TASK_B = ("The folder sales/ holds one CSV per region. Write report.md with "
          "the total amount per region and a grand total. Compute the numbers "
          "by running Python, do not estimate them.")

make_workspace()
print(run_agent(TASK_B))
print(read_file("report.md"))

It usually writes itself a script, runs it, and writes the report from the
output. Sometimes it tries to run a file that does not exist first, gets an
error back, and carries on. That recovery only works because the error went
back to the model as a result instead of stopping the loop.

## Bounding it

Three turns is not enough for this task. The harness stops and says so,
instead of running on until the bill arrives.

In [ ]:
make_workspace()
run_agent(TASK_B, max_turns=3)

## Asking first

Two of the four tools change things. An approval hook sees every call before
it runs. This one lets the model look and run, but not change a file.

In [ ]:
def look_but_do_not_touch(name, args):
    return name != "write_file"


make_workspace()
print(run_agent(TASK_A, approve=look_but_do_not_touch))

It finds the bug, is refused the fix, and comes back with the exact change
for us to make. That is the human in the loop from module 09, and in front of
a person the hook is one line different :

```python
def ask_me(name, args):
    return input(f"{name}({brief(args)}) ? [y/N] ").lower() == "y"
```

It is left out here so that *Run all* never sits waiting for a keypress.

The wording of the refusal matters. Told only *not approved, do not retry*, the model writes a wrapper script and tries to run
the code another way, until the turn limit stops it. A one-line hook is a
demonstration ; it is not a permission system.

## What it cost

The meter printed it. Each run is about six turns and five to six thousand
tokens. Almost all of that is the
conversation being re-sent, which is why demos04f and exercise27 spend their
time on what goes into it.

## What this is, and is not

Those forty lines are the shape of every coding agent : a loop, a tool list,
and results going back as messages. What Claude Code, Codex and the rest add
sits on top of that shape :

- streaming, so we see the answer arrive
- a permission system instead of a one-line hook
- compaction, so a long conversation does not fill the window
- sub-agents, which are this same loop started with a narrower task
- MCP, so the tool list comes from servers instead of from a dictionary

Each of those is a slide somewhere in modules 05 to 09. And `run_python` is a
shell : it runs whatever the model wrote, inside this machine, on these files.
That is the point of the demo, and it is also the whole argument for the
approval hook.

## Things to try

Nothing here is required.

- Rename `run_python` to `tool_3` in `SCHEMA` and `TOOLS`, and blank its
  description. Run `TASK_A` again. Does it still check its work?
- Set `max_turns=1` and read what the model says when it is cut off
- Put a second bug in `PROGRAM` and see whether one run finds both
- Give it a task the tools cannot do : *"email the report to finance"*